# 14.3 提示优化 (Prompt Optimization)

自动优化提示文本以获得更好的模型输出。

本节涵盖：自动提示工程（APE）、软提示优化、提示鲁棒性

## 1. 自动提示工程与软提示优化

**APE（Automatic Prompt Engineer）**：自动搜索最优提示文本，通过生成-评估循环找到最佳提示。

**软提示优化**：将提示表示为连续向量（非文本），通过梯度优化找到最优提示嵌入。

**与Prompt Tuning的区别**：
- Prompt Tuning：在输入嵌入层添加可学习向量
- 软提示优化：优化整个提示的嵌入表示

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SoftPromptOptimizer(nn.Module):
    def __init__(self, d=64, n_soft_tokens=5, vocab_size=100):
        super().__init__()
        self.soft_prompt = nn.Parameter(torch.randn(n_soft_tokens, d) * 0.02)
        self.embed = nn.Embedding(vocab_size, d)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, 2, d * 4, batch_first=True), 1
        )
        self.head = nn.Linear(d, vocab_size)

    def forward(self, x):
        B = x.shape[0]
        txt_embeds = self.embed(x)
        soft_tokens = self.soft_prompt.unsqueeze(0).expand(B, -1, -1)
        combined = torch.cat([soft_tokens, txt_embeds], dim=1)
        h = self.transformer(combined)
        return self.head(h[:, self.soft_prompt.shape[0]:])

model = SoftPromptOptimizer(n_soft_tokens=5)
x = torch.randint(0, 100, (2, 10))
out = model(x)

print('=== Soft Prompt Optimization ===')
print(f'Input tokens: {x.shape}')
print(f'Soft prompt: {model.soft_prompt.shape} (learnable)')
print(f'Combined input: {x.shape[0] + model.soft_prompt.shape[0]} tokens')
print(f'Output: {out.shape}')
print(f'\nKey: Soft prompts are continuous vectors optimized by gradient descent.')
print(f'They can find better representations than discrete text prompts.')